# 🎛️ Demucs → Timbre Space (high-quality stems)

Runs **Demucs** (Meta's state-of-the-art source separation) in Colab to produce cleaner instrument stems than the in-repo Spleeter pipeline, then extracts the same timbre features and exports the **exact files the WebGL visualization consumes**:

- `web/timbre_data.json` — points + per-stem centroids + fly-through order
- `web/audio/<stem>.ogg` — separated stems for in-browser playback

Features per onset/frame: **X = Attack time**, **Y = Brightness** (spectral centroid), **Z = Spectral flux**.

**How to use:** `Runtime → Run all`. Upload your own song when prompted (or skip to use a sample). At the end, download `timbre_space_data.zip` and drop its `web/` folder into the repo root (replacing the existing one), then commit — GitHub Pages redeploys automatically.

💡 For best speed pick a GPU runtime: `Runtime → Change runtime type → T4 GPU`.

In [ ]:
#@title 1 · Install Demucs + audio libs
!pip -q install -U demucs soundfile librosa 2>/dev/null
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

In [ ]:
#@title 2 · Config
# htdemucs_6s -> 6 stems (drums, bass, other, vocals, guitar, piano)
# htdemucs    -> 4 stems (drums, bass, other, vocals)  [highest fidelity]
MODEL_NAME = 'htdemucs_6s'  #@param ['htdemucs_6s', 'htdemucs', 'htdemucs_ft']
POINTS_PER_STEM = 650       #@param {type:'integer'}
ANALYSIS_SR = 22050         # feature/export rate

# per-stem display colours (matches the viz legend; unknown stems get a default)
STEM_COLORS = {
    'vocals':'#ff5e9c', 'piano':'#5be0ff', 'drums':'#ffb347',
    'bass':'#9b7bff', 'other':'#7cff8a', 'guitar':'#ffd45e',
}
DEFAULT_COLOR = '#9fd0ff'

In [ ]:
#@title 3 · Get audio (upload your own, or use a sample)
import os, soundfile as sf, numpy as np
os.makedirs('in', exist_ok=True)
audio_path = None
try:
    from google.colab import files
    print('Upload an audio file (mp3/wav/flac/ogg) — or click Cancel to use a sample.')
    up = files.upload()
    if up:
        audio_path = os.path.join('in', list(up.keys())[0])
        with open(audio_path, 'wb') as f: f.write(list(up.values())[0])
except Exception as e:
    print('(upload skipped:', e, ')')

if not audio_path:
    import urllib.request
    audio_path = 'in/example.mp3'
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/deezer/spleeter/master/audio_example.mp3', audio_path)
    print('Using sample clip.')

SAMPLE_NAME = os.path.basename(audio_path)
info = sf.info(audio_path)
print('Audio:', SAMPLE_NAME, '|', round(info.duration,1), 's |', info.samplerate, 'Hz |', info.channels, 'ch')

In [ ]:
#@title 4 · Separate with Demucs
import torch, torchaudio, librosa, numpy as np
from demucs.pretrained import get_model
from demucs.apply import apply_model

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = get_model(MODEL_NAME); model.to(device).eval()
msr = model.samplerate  # demucs expects 44100
print('model:', MODEL_NAME, '| stems:', model.sources, '| sr:', msr)

# load -> (channels, samples) @ model samplerate, stereo
wav, sr = torchaudio.load(audio_path)
if sr != msr:
    wav = torchaudio.functional.resample(wav, sr, msr)
if wav.shape[0] == 1:
    wav = wav.repeat(2, 1)
ref = wav.mean(0)
wav = (wav - ref.mean()) / (ref.std() + 1e-8)

with torch.no_grad():
    sources = apply_model(model, wav[None].to(device), device=device,
                          split=True, overlap=0.25, progress=True)[0]
sources = sources * ref.std() + ref.mean()

stems = {}
for name, src in zip(model.sources, sources.cpu().numpy()):
    mono = src.mean(axis=0).astype('float32')          # downmix
    mono = librosa.resample(mono, orig_sr=msr, target_sr=ANALYSIS_SR)
    stems[name] = mono
print('separated stems:', list(stems.keys()))

In [ ]:
#@title 5 · Extract timbre features (attack / brightness / flux)
import numpy as np, librosa
np.random.seed(7)

def stem_features(y, sr=ANALYSIS_SR, n_points=POINTS_PER_STEM):
    hop = 512
    S = np.abs(librosa.stft(y, n_fft=2048, hop_length=hop))
    centroid = librosa.feature.spectral_centroid(S=S, sr=sr, hop_length=hop)[0]
    flux = librosa.onset.onset_strength(
        S=librosa.amplitude_to_db(S, ref=np.max), sr=sr, hop_length=hop)
    rms = librosa.feature.rms(S=S, hop_length=hop)[0]
    n = min(len(centroid), len(flux), len(rms))
    centroid, flux, rms = centroid[:n], flux[:n], rms[:n]
    onsets = list(librosa.onset.onset_detect(
        onset_envelope=flux, sr=sr, hop_length=hop, backtrack=True)) + [n]
    attack = np.zeros(n, 'float32')
    for i in range(len(onsets)-1):
        a, b = onsets[i], min(onsets[i]+20, onsets[i+1])
        if b > a:
            peak = a + int(np.argmax(rms[a:b]))
            attack[a:onsets[i+1]] = np.clip((peak-a)*hop/sr*1000.0, 1.0, 160.0)
    keep = np.where(rms > max(rms.mean()*0.6, 1e-4))[0]
    if len(keep) == 0: keep = np.arange(n)
    if len(keep) > n_points: keep = np.random.choice(keep, n_points, replace=False)
    return attack[keep], centroid[keep], flux[keep], rms[keep]

raw = {name: dict(zip(['atk','bright','flux','rms'], stem_features(y))) for name, y in stems.items()}
for name, r in raw.items():
    print(f"  {name:7s}: {len(r['atk'])} pts  attack~{np.median(r['atk']):.0f}ms  bright~{np.median(r['bright']):.0f}Hz")

In [ ]:
#@title 6 · Build 3D layout + export web/timbre_data.json and web/audio/*.ogg
import os, json, numpy as np, soundfile as sf
os.makedirs('web/audio', exist_ok=True)
names = list(raw.keys()); nS = len(names)

# write stems
for name, y in stems.items():
    sf.write(f'web/audio/{name}.ogg', y, ANALYSIS_SR, format='OGG', subtype='VORBIS')

# global per-axis normalisation
all_atk = np.concatenate([raw[n]['atk'] for n in names])
all_brt = np.log1p(np.concatenate([raw[n]['bright'] for n in names]))
all_flx = np.concatenate([raw[n]['flux'] for n in names])
def sc(a, lo=2, hi=98):
    l, h = np.percentile(a, lo), np.percentile(a, hi); return l, (h-l if h>l else 1.0)
a_lo,a_rng = sc(all_atk); b_lo,b_rng = sc(all_brt); f_lo,f_rng = sc(all_flx)
def nm(v, lo, rng, span): return (np.clip((v-lo)/rng,0,1)-0.5)*2*span

# spread stems on a circle in XZ so separated instruments form distinct galaxies
lanes = {n:(24*np.cos(2*np.pi*i/nS), 24*np.sin(2*np.pi*i/nS)) for i,n in enumerate(names)}

stems_meta, points = [], []
for si, name in enumerate(names):
    r = raw[name]
    x = nm(r['atk'], a_lo, a_rng, 26.0)
    y = nm(np.log1p(r['bright']), b_lo, b_rng, 16.0)
    z = nm(r['flux'], f_lo, f_rng, 26.0)
    lx, lz = lanes[name]; bias = 0.55
    x = x*(1-bias) + lx*bias + np.random.uniform(-3,3,len(x))
    z = z*(1-bias) + lz*bias + np.random.uniform(-3,3,len(z))
    cx, cy, cz = float(np.mean(x)), float(np.mean(y)), float(np.mean(z))
    stems_meta.append({'name':name, 'color':STEM_COLORS.get(name, DEFAULT_COLOR),
                       'centroid':[cx,cy,cz], 'count':int(len(x)),
                       'meanBrightnessHz':float(np.median(r['bright'])),
                       'audio':f'web/audio/{name}.ogg'})
    rmsn = r['rms']/(r['rms'].max()+1e-9)
    for i in range(len(x)):
        points.append([round(float(x[i]),2), round(float(y[i]),2), round(float(z[i]),2), si, round(float(rmsn[i]),2)])

order = sorted(range(nS), key=lambda i: stems_meta[i]['meanBrightnessHz'])
data = {'sample': f'{SAMPLE_NAME} (Demucs {MODEL_NAME})',
        'method': f'Demucs {MODEL_NAME} (high-quality source separation)',
        'axes': {'x':'Attack Time','y':'Brightness (spectral centroid)','z':'Spectral Flux'},
        'stems': stems_meta, 'trajectory': order, 'points': points,
        'pointFormat': ['x','y','z','stemIndex','rms']}
with open('web/timbre_data.json','w') as f: json.dump(data, f, separators=(',',':'))
print(f'wrote web/timbre_data.json  ({len(points)} points, {nS} stems)')
print('stems:', ', '.join(s['name'] for s in stems_meta))

In [ ]:
#@title 7 · Download results
import shutil
shutil.make_archive('timbre_space_data', 'zip', '.', 'web')
print('Created timbre_space_data.zip — unzip into the repo root (replaces web/), then commit & push.')
try:
    from google.colab import files
    files.download('timbre_space_data.zip')
except Exception as e:
    print('(manual download — see the Files panel)', e)

## Putting it in the repo

1. Unzip `timbre_space_data.zip` — it contains a `web/` folder.
2. Replace the repo's existing `web/` with it (so `web/timbre_data.json` and `web/audio/*.ogg` are updated).
3. Commit & push to `main`:
   ```bash
   git add web && git commit -m "Update timbre data with Demucs stems" && git push
   ```
4. GitHub Pages redeploys automatically → https://megan8821.github.io/audio_generation/

The visualization is data-driven, so it picks up the new stems, colours, clusters, and audio with no code changes.